# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR² dataset "Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution" using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Install `mlcroissant` if not already available
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Croissant metadata and dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset title and description by accessing as attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review the available record sets in the dataset, listing their `@id`s along with their names and fields.
All references to entities (record sets, fields, columns) are by their `@id` as per FAIR2/Croissant best practices.

In [ ]:
# List all record sets and their fields
record_sets = list(dataset.record_sets())
if not record_sets:
    print("No record sets found in this dataset. The dataset may encode only one record set, or the schema is non-tabular.")
else:
    for rs in record_sets:
        print(f"Record set ID: {rs['@id']}")
        print(f"  Name: {rs.get('name', '(unnamed)')}")
        print(f"  Fields (by @id):")
        for field in rs.get('field', []):
            print(f"    - {field['@id']} (name: {field.get('name', '')})")
        print('')

# For demonstration, try listing first few records from the (first) record set by @id
if record_sets:
    first_rs_id = record_sets[0]['@id']
    print(f"\nExample records from RecordSet {first_rs_id}:")
    for i, rec in enumerate(dataset.records(record_set=first_rs_id)):
        if i > 2:
            break
        print(rec)


## 3. Data Extraction
Load data from each record set into a `pandas.DataFrame` for analysis. Use the `@id` of each record set.

In [ ]:
# Get list of record set @id's
record_sets_list = [rs['@id'] for rs in dataset.record_sets()]

dataframes = {}
for rs_id in record_sets_list:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[rs_id])} records for RecordSet {rs_id}")

if dataframes:
    main_record_set_id = record_sets_list[0]
    print(f"Column names in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping using record set and field `@id`s.

*For demonstration, select the first available numeric field. You may adjust the field by its `@id` according to the dataset schema.*

In [ ]:
# Identify numeric fields (by Croissant @id) in the first record set
main_rs = None
for rs in dataset.record_sets():
    if rs['@id'] == main_record_set_id:
        main_rs = rs
        break
numeric_field_id = None
if main_rs:
    for field in main_rs.get('field', []):
        dt = field.get('dataType', '').lower()
        if 'int' in dt or 'float' in dt or dt == 'number' or dt == 'schema:number' or dt == 'schema:integer' or dt == 'schema:float':
            numeric_field_id = field['@id']
            print(f"Using numeric field: {numeric_field_id} ({field.get('name','')})")
            break
if not numeric_field_id:
    print("No numeric field found. You may need to adjust numeric_field_id manually.")

# Example threshold for filtering
threshold = 10
df = dataframes[main_record_set_id]

if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id].astype(float) > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by another field, e.g., first non-numeric field
    group_field = None
    for field in main_rs.get('field', []):
        if field['@id'] != numeric_field_id:
            # Attempt to use first textual/categorical field
            group_field = field['@id']
            print(f"Grouping by field: {group_field}")
            break
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in DataFrame columns.")

## 5. Visualization
Visualize numeric field values and their distributions, and relationships between numeric and group fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].astype(float), bins=12, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()

## 6. Conclusion

In this notebook, we've:
- Loaded the metadata and records for the FAIR² dataset on second primary colorectal cancer.
- Explored available record sets, fields, and reviewed the structure via Croissant `@id`s.
- Extracted tabular data using `mlcroissant` and performed basic statistical and group-wise analysis.
- Visualized distributions and category-based differences for a chosen numeric field.

Further analysis can be performed according to your research needs, exploring additional fields and relationships.
